# inf-masking — faded example 2: Fill the pad-mask broadcast

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `inf-masking`. Running the beacon reports progress on the `Numpy: Inf-fill masking trick` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Inf-fill masking trick` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inf-masking`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inf-masking"
DD_SUBTOPIC = "Numpy: Inf-fill masking trick"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

A `(B, S_k)` key padding mask must be reshaped to `(B, 1, S_k)` so it broadcasts across all query rows before `masked_fill(-inf)` is applied to the `(B, S_q, S_k)` scores.

## Faded exercise 2

Implement `pad_mask_attention(scores, pad_mask)`. `scores` is `(B, S_q, S_k)`, `pad_mask` is `(B, S_k)`. Broadcast the mask across queries, fill with `-inf`, and softmax. Complete the blanked masked-fill using the broadcast mask.

**Fill in:** the masked_fill that broadcasts pad_mask to (B, 1, S_k) and fills with -inf

In [ ]:
import torch as t

t.manual_seed(4)
scores = t.randn(2, 3, 4)
pad_mask = t.tensor([[False, False, True, False], [True, False, False, True]])

def pad_mask_attention(scores, pad_mask):
    masked = scores.masked_fill(pad_mask.unsqueeze(1), float('-inf'))
    return masked.softmax(dim=-1)

print(pad_mask_attention(scores, pad_mask).shape)


def _test():
    w = pad_mask_attention(scores, pad_mask)
    assert w.shape == (2, 3, 4)
    # PAD columns get exactly zero mass, for every query row, per batch element
    for b in range(2):
        for j in range(4):
            if bool(pad_mask[b, j]):
                assert bool((w[b, :, j] == 0).all()), (b, j)
    # each query row sums to 1 over the surviving keys
    assert t.allclose(w.sum(-1), t.ones(2, 3), atol=1e-6)


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(4)
scores = t.randn(2, 3, 4)
pad_mask = t.tensor([[False, False, True, False], [True, False, False, True]])

def pad_mask_attention(scores, pad_mask):
    masked = scores.masked_fill(pad_mask.unsqueeze(1), float('-inf'))
    return masked.softmax(dim=-1)

print(pad_mask_attention(scores, pad_mask).shape)
```
</details>